<a href="https://colab.research.google.com/github/Pujitha-pitta/Flyrank-ML-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [32]:

%pip -q install duckdb huggingface_hub scikit-learn pandas

In [33]:
HF_TOKEN = "HF_TOKEN"

In [34]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
print(api.whoami())

{'type': 'user', 'id': '6a5f81919c6b339e0c6b3608', 'name': 'PujithaPitta', 'fullname': 'NAGA PUJITHA', 'isPro': False, 'avatarUrl': '/avatars/80e071f7399fd5ce3d17edc4432fe7d1.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'flyrank', 'role': 'fineGrained', 'createdAt': '2026-07-21T14:32:35.003Z', 'fineGrained': {'canReadGatedRepos': True, 'global': [], 'scoped': [{'entity': {'_id': '6a5f81919c6b339e0c6b3608', 'type': 'user', 'name': 'PujithaPitta'}, 'permissions': ['repo.content.read']}]}}}}


In [35]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)

files = api.list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset"
)

print("Number of files:", len(files))
print(files[:10])  # Show the first 10 files

Number of files: 24
['.gitattributes', 'README.md', 'dim_clients.parquet', 'dim_content.parquet', 'fact_content_daily_performance/month=2025-01/data_0.parquet', 'fact_content_daily_performance/month=2025-02/data_0.parquet', 'fact_content_daily_performance/month=2025-03/data_0.parquet', 'fact_content_daily_performance/month=2025-04/data_0.parquet', 'fact_content_daily_performance/month=2025-05/data_0.parquet', 'fact_content_daily_performance/month=2025-06/data_0.parquet']


In [36]:
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
);
""")

In [37]:
daily = """
read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet',
hive_partitioning=true
)
"""

In [38]:
query = f"""
SELECT *
FROM {daily}
LIMIT 5
"""

df = con.sql(query).df()

df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,2025-01
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,2025-01
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,2025-01
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,2025-01
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,2025-01


In [39]:
print(df.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [40]:
con.sql(f"""
SELECT DISTINCT month
FROM {daily}
ORDER BY month
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,month
0,2025-01
1,2025-02
2,2025-03
3,2025-04
4,2025-05
5,2025-06
6,2025-07
7,2025-08
8,2025-09
9,2025-10


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*



The source warehouse table contains one row for one pseudonymized client-content pair on one report date.

For my Search Intelligence modelling frame, I aggregate the daily records so that one row represents one pseudonymized webpage for one client during March 2026.

- Feature window: 1 March 2026 to 31 March 2026
- Outcome window: 1 April 2026 to 30 April 2026
- Decision: rank webpages that may need a content refresh
- Proxy label: needs_refresh_proxy = 1 when April search clicks are lower than March search clicks; otherwise `0`

This is a decision-support proxy. A decline in clicks does not prove that a page needs refreshing because seasonality, search demand, competitors, and algorithm changes may also affect performance.

I deliberately exclude April performance values, the proxy label, and any columns derived from the label from the feature set because they would not be knowable at the March decision moment.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*



**features **

1. `march_impressions` – total GSC impressions observed during March 2026.
2. `march_clicks` – total GSC clicks observed during March 2026.
3. `march_avg_position` – impression-weighted average search position during March 2026.
4. `march_sessions` – total GA4 sessions observed during March 2026.
5. `march_engagement_rate` – engaged sessions divided by total sessions during March 2026.

**Label**

`needs_refresh_proxy`

- 1: April clicks are lower than March clicks.
- 0: April clicks are equal to or higher than March clicks.

**Context fields**

- `client_hash_id`
- `content_hash_id`
- `feature_month`

These identify the observation but are not used as model features.

**Excluded fields**

- `april_clicks`: future outcome information.
- `needs_refresh_proxy`: the target itself.
- Any direct copy or transformation of the label: target leakage.
- Raw client and content identifiers: identifiers do not represent transferable webpage behaviour.
- AI-source fields: excluded to keep this first feature frame limited to five features.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [41]:
grain_query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(
        DISTINCT (client_hash_id, content_hash_id, report_date)
    ) AS unique_client_content_dates,
    COUNT(*) -
    COUNT(
        DISTINCT (client_hash_id, content_hash_id, report_date)
    ) AS duplicate_rows
FROM {daily}
WHERE month = '2026-03'
"""

grain_result = con.sql(grain_query).df()
grain_result

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_client_content_dates,duplicate_rows
0,9841378,9841378,0


The number of total rows matches the number of unique client-content-date combinations if duplicate_rows is zero. This supports the claim that the source table has one row per client, content item, and report date.

In [27]:
count_span_query = f"""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS content_items,
    MIN(report_date) AS first_report_date,
    MAX(report_date) AS last_report_date
FROM {daily}
WHERE month = '2026-03'
"""

count_span_result = con.sql(count_span_query).df()
count_span_result

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,clients,content_items,first_report_date,last_report_date
0,9841378,55,331437,2026-03-01,2026-03-31


This query measures the size of the March 2026 slice and verifies its observed date range.

In [28]:
availability_query = f"""
WITH all_march_rows AS (
    SELECT COUNT(*) AS total_rows
    FROM {daily}
    WHERE month = '2026-03'
),
available_march_rows AS (
    SELECT COUNT(*) AS available_rows
    FROM {daily}
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
)
SELECT
    total_rows,
    available_rows,
    ROUND(
        100.0 * available_rows / NULLIF(total_rows, 0),
        2
    ) AS percent_surviving
FROM all_march_rows
CROSS JOIN available_march_rows
"""

availability_result = con.sql(availability_query).df()
availability_result

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,available_rows,percent_surviving
0,9841378,364347,3.7


Only rows where both GSC and GA4 data are marked as available are retained for the combined search-and-engagement feature frame. The percentage shows how much of the original March slice survives this requirement.

In [42]:
feature_frame_query = f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(COALESCE(gsc_impressions, 0)) AS march_impressions,
        SUM(COALESCE(gsc_clicks, 0)) AS march_clicks,

        SUM(COALESCE(gsc_sum_position, 0))
        / NULLIF(SUM(COALESCE(gsc_impressions, 0)), 0)
        AS march_avg_position,

        SUM(COALESCE(ga4_sessions, 0)) AS march_sessions,

        SUM(COALESCE(ga4_engaged_sessions, 0))
        / NULLIF(SUM(COALESCE(ga4_sessions, 0)), 0)
        AS march_engagement_rate

    FROM {daily}
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),

april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(COALESCE(gsc_clicks, 0)) AS april_clicks
    FROM {daily}
    WHERE month = '2026-04'
      AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    m.client_hash_id,
    m.content_hash_id,
    '2026-03' AS feature_month,

    m.march_impressions,
    m.march_clicks,
    m.march_avg_position,
    m.march_sessions,
    m.march_engagement_rate,

    a.april_clicks,

    CASE
        WHEN a.april_clicks < m.march_clicks THEN 1
        ELSE 0
    END AS needs_refresh_proxy

FROM march AS m
INNER JOIN april AS a
    ON m.client_hash_id = a.client_hash_id
   AND m.content_hash_id = a.content_hash_id
"""

feature_frame = con.sql(feature_frame_query).df()

print("Feature-frame shape:", feature_frame.shape)
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature-frame shape: (62681, 10)


,client_hash_id,content_hash_id,feature_month,march_impressions,march_clicks,march_avg_position,march_sessions,march_engagement_rate,april_clicks,needs_refresh_proxy
0,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,2026-03,458.0,2.0,4.338428,14.0,0.000000,8.0,0
1,client_65de48885f4ef01b,content_e25ea7297a1dffd3,2026-03,3943.0,23.0,4.291149,54.0,0.018519,16.0,1
2,client_65de48885f4ef01b,content_3c286ded8bd68120,2026-03,2180.0,15.0,8.418807,30.0,0.066667,7.0,1
3,client_65de48885f4ef01b,content_b2108e8fe3360fa6,2026-03,503.0,8.0,5.727634,23.0,0.043478,1.0,1
4,client_65de48885f4ef01b,content_ff867882e604fa96,2026-03,24.0,0.0,3.583333,2.0,0.000000,4.0,0


In [29]:
feature_frame["needs_refresh_proxy"].value_counts(dropna=False)

,count
needs_refresh_proxy,
0,45806
1,16875


In [30]:
feature_columns = [
    "march_impressions",
    "march_clicks",
    "march_avg_position",
    "march_sessions",
    "march_engagement_rate",
]

feature_frame[feature_columns].isna().sum().to_frame("missing_values")

,missing_values
march_impressions,0
march_clicks,0
march_avg_position,0
march_sessions,0
march_engagement_rate,199


**Feature availability at the decision moment**

- `march_impressions`: knowable because March Search Console impressions have already been observed when the March reporting window closes.
- `march_clicks`: knowable because March Search Console clicks have already been recorded.
- `march_avg_position`: knowable because it is calculated only from March impressions and position totals.
- `march_sessions`: knowable because March GA4 sessions have already been measured.
- `march_engagement_rate`: knowable because it is calculated only from March sessions and engaged sessions.

April clicks are not a feature. They are used only to construct the later outcome proxy.

In [31]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [43]:
X_honest = feature_frame[feature_columns].copy()
y = feature_frame["needs_refresh_proxy"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X_honest,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42
            )
        ),
    ]
)

model.fit(X_train, y_train)

honest_predictions = model.predict(X_test)
honest_probabilities = model.predict_proba(X_test)[:, 1]

honest_results = {
    "accuracy": accuracy_score(y_test, honest_predictions),
    "f1": f1_score(y_test, honest_predictions),
    "roc_auc": roc_auc_score(y_test, honest_probabilities),
}

honest_results

{'accuracy': 0.6855976006636463,
 'f1': 0.3877221324717286,
 'roc_auc': np.float64(0.6383604946668999)}

In [44]:
# Deliberate leakage experiment:
# this column is a direct copy of the target and must never be retained.

X_leaked = feature_frame[feature_columns].copy()
X_leaked["label_derived_leak"] = feature_frame["needs_refresh_proxy"]

X_train_leak, X_test_leak, y_train_leak, y_test_leak = train_test_split(
    X_leaked,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

leaked_model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42
            )
        ),
    ]
)

leaked_model.fit(X_train_leak, y_train_leak)

leaked_predictions = leaked_model.predict(X_test_leak)
leaked_probabilities = leaked_model.predict_proba(X_test_leak)[:, 1]

leaked_results = {
    "accuracy": accuracy_score(y_test_leak, leaked_predictions),
    "f1": f1_score(y_test_leak, leaked_predictions),
    "roc_auc": roc_auc_score(y_test_leak, leaked_probabilities),
}

leaked_results

{'accuracy': 1.0, 'f1': 1.0, 'roc_auc': np.float64(1.0)}

In [45]:
comparison = pd.DataFrame(
    [
        {"experiment": "Honest five-feature model", **honest_results},
        {"experiment": "Model with label-derived leakage", **leaked_results},
    ]
)

comparison

,experiment,accuracy,f1,roc_auc
0,Honest five-feature model,0.685598,0.387722,0.63836
1,Model with label-derived leakage,1.000000,1.000000,1.00000


In [46]:
X_final = X_leaked.drop(columns=["label_derived_leak"])

assert "label_derived_leak" not in X_final.columns
assert list(X_final.columns) == feature_columns

print("Leakage removed.")
print("Final features:", X_final.columns.tolist())
print("Number of final features:", X_final.shape[1])

Leakage removed.
Final features: ['march_impressions', 'march_clicks', 'march_avg_position', 'march_sessions', 'march_engagement_rate']
Number of final features: 5


**Leakage lesson**

The score increased toward perfect after I added `label_derived_leak`, because this column directly revealed the correct target to the model. This was not genuine predictive performance.

I removed the leaked column and retained the honest five-feature result. The lower honest score is more credible because it uses only information observed during the March decision window.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

4. Data limits

This data can measure recorded search and engagement behaviour, but it cannot prove why performance changed.

Important limitations include:

- The proxy treats lower April clicks as refresh need, although clicks may decline because of seasonality, lower search demand, competitor activity, or search-engine changes.
- Only content appearing in both March and April enters the labelled frame. New, removed, or unavailable content is excluded.
- Requiring both GSC and GA4 availability may create a more complete but less representative subset.
- Some earlier historical rows may contain GSC data without equivalent GA4 coverage.
- Adjacent monthly windows may contain related behaviour, so the outcome is not fully independent of the feature month.
- A single month may not represent long-term performance.
- This analysis supports human prioritisation; it should not automatically trigger a content refresh.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.